In [1]:
import pandas as pd
import numpy as np
import os

from pathlib import Path
from difflib import SequenceMatcher

In [2]:
output_path = Path("data")

output_path.mkdir(exist_ok=True)

print("Output folder:", output_path.resolve())

Output folder: C:\Apache Spark Environment Programs\data


In [4]:
data_path = "data"

customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(data_path, "olist_geolocation_dataset.csv"))
order_items = pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(data_path, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(data_path, "olist_order_reviews_dataset.csv"))
orders = pd.read_csv(os.path.join(data_path, "olist_orders_dataset.csv"))
products = pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(data_path, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(os.path.join(data_path, "product_category_name_translation.csv"))

In [5]:
dataframes = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

In [6]:
clean_dataframes = {
    name: df.copy(deep=True)
    for name, df in dataframes.items()
}

print("Original tables:", len(dataframes))
print("Working tables:", len(clean_dataframes))

Original tables: 9
Working tables: 9


In [7]:
g = clean_dataframes["geolocation"].copy()

print("Original Rows:", len(g))

print("Exact Duplicate Rows:", g.duplicated().sum())

print("Invalid Latitude:",
      (~g["geolocation_lat"].between(-33.8, 5.3)).sum())

print("Invalid Longitude:",
      (~g["geolocation_lng"].between(-73.99, -34.7)).sum())

print("Missing Values:", g.isnull().sum().sum())

Original Rows: 1000163
Exact Duplicate Rows: 261831
Invalid Latitude: 31
Invalid Longitude: 37
Missing Values: 0


In [8]:
g = g.drop_duplicates()

g = g[
    g["geolocation_lat"].between(-33.8, 5.3) &
    g["geolocation_lng"].between(-73.99, -34.7)
].copy()

print("Rows Before Cleaning: 1000163")
print("Rows After Cleaning:", len(g))
print("Remaining Duplicates:", g.duplicated().sum())

Rows Before Cleaning: 1000163
Rows After Cleaning: 738299
Remaining Duplicates: 0


In [9]:
def clean_city(s):
    s = s.astype("string")

    s = s.str.replace(
        r"%26apos%3b|[´`]",
        "'",
        regex=True,
        case=False
    )

    s = s.str.normalize("NFKD")
    s = s.str.encode("ascii", errors="ignore").str.decode("ascii")
    s = s.str.lower().str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)

    return s

g["geolocation_city"] = clean_city(g["geolocation_city"])

g["geolocation_city"] = g["geolocation_city"].replace(
    {"saopaulo": "sao paulo"}
)

In [10]:
print("Final Rows:", len(g))
print("Duplicates:", g.duplicated().sum())
print("Missing Values:", g.isnull().sum().sum())

clean_dataframes["geolocation"] = g

g.to_csv(
    output_path / "olist_geolocation_clean.csv",
    index=False
)

Final Rows: 738299
Duplicates: 17837
Missing Values: 0


In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. PATHS
# ==========================================

data_path = Path("data")
output_path = Path("cleaned_data")
output_path.mkdir(exist_ok=True)

# ==========================================
# 2. LOAD ALL TABLES
# ==========================================

dtype_config = {
    "geolocation_zip_code_prefix": "string",
    "customer_zip_code_prefix": "string",
    "seller_zip_code_prefix": "string"
}

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

clean_dataframes = {}

for name, filename in files.items():
    clean_dataframes[name] = pd.read_csv(
        data_path / filename,
        dtype=dtype_config
    )

print("All tables loaded:", len(clean_dataframes))


# ==========================================
# 3. CITY CLEANING FUNCTION
# ==========================================

def clean_city(s):

    s = s.astype("string")

    s = s.str.replace(
        r"%26apos%3b|[´`]",
        "'",
        regex=True,
        case=False
    )

    s = s.str.normalize("NFKD")

    s = s.str.encode(
        "ascii",
        errors="ignore"
    ).str.decode("ascii")

    s = s.str.lower().str.strip()

    s = s.str.replace(
        r"\s+",
        " ",
        regex=True
    )

    return s
# ==========================================
# 4. GEOLOCATION CLEANING
# ==========================================

g = clean_dataframes["geolocation"].copy()

g = g[
    g["geolocation_lat"].between(-33.8, 5.3) &
    g["geolocation_lng"].between(-73.99, -34.7)
].copy()

g["geolocation_city"] = clean_city(
    g["geolocation_city"]
)

g["geolocation_city"] = g["geolocation_city"].replace(
    {"saopaulo": "sao paulo"}
)

# Remove exact duplicates after standardization
g = g.drop_duplicates().reset_index(drop=True)

clean_dataframes["geolocation"] = g


# ==========================================
# 5. SELLERS CLEANING
# ==========================================

s = clean_dataframes["sellers"].copy()

s["seller_city"] = clean_city(s["seller_city"])

clean_dataframes["sellers"] = s


# ==========================================
# 6. CUSTOMERS CLEANING
# ==========================================

c = clean_dataframes["customers"].copy()

c["customer_city"] = clean_city(c["customer_city"])

clean_dataframes["customers"] = c


# ==========================================
# 7. PRODUCTS CLEANING
# ==========================================

pr = clean_dataframes["products"].copy()

# Correct column spelling
pr = pr.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

# Zero physical measurements are treated as missing
size_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

pr[size_columns] = pr[size_columns].replace(0, np.nan)

clean_dataframes["products"] = pr


# ==========================================
# 8. CATEGORY TRANSLATION
# ==========================================

t = clean_dataframes["category_translation"].copy()

known_translations = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "portable_kitchen_food_preparers"
}

for category, translation in known_translations.items():

    if (
        category in pr["product_category_name"].dropna().values
        and category not in t["product_category_name"].values
    ):

        t.loc[len(t)] = [category, translation]

clean_dataframes["category_translation"] = t


# ==========================================
# 9. KEEP OTHER TABLES WITHOUT DATA DELETION
# ==========================================

# Orders: preserve missing dates
# Reviews: preserve missing comments
# Payments: preserve all payment records
# Order Items: preserve all records


# ==========================================
# 10. QUALITY REPORT
# ==========================================

quality_report = []

for name, df in clean_dataframes.items():

    quality_report.append({

        "Table": name,

        "Rows": len(df),

        "Columns": len(df.columns),

        "Exact_Duplicates": df.duplicated().sum(),

        "Missing_Values": df.isnull().sum().sum(),

        "Duplicate_Percentage": round(
            df.duplicated().sum() / len(df) * 100, 2
        )

    })

quality_df = pd.DataFrame(quality_report)

print("\nFINAL QUALITY REPORT")

display(quality_df)


# ==========================================
# 11. SAVE CLEANED CSV FILES
# ==========================================

for name, df in clean_dataframes.items():

    df.to_csv(
        output_path / f"olist_{name}_clean.csv",
        index=False
    )

# Save quality report
quality_df.to_csv(
    output_path / "data_quality_report.csv",
    index=False
)


# ==========================================
# 12. FINAL RESULT
# ==========================================

print("\nCleaning Completed!")

print("Output Folder:", output_path.resolve())

print("\nGenerated Files:")

for file in sorted(output_path.iterdir()):
    print(file.name)

All tables loaded: 9

FINAL QUALITY REPORT


,Table,Rows,Columns,Exact_Duplicates,Missing_Values,Duplicate_Percentage
0,customers,99441,5,0,0,0.0
1,geolocation,720462,5,0,0,0.0
2,order_items,112650,7,0,0,0.0
3,order_payments,103886,5,0,0,0.0
4,order_reviews,99224,7,0,145903,0.0
5,orders,99441,8,0,4908,0.0
6,products,32951,9,0,2452,0.0
7,sellers,3095,4,0,0,0.0
8,category_translation,73,2,0,0,0.0



Cleaning Completed!
Output Folder: C:\Apache Spark Environment Programs\cleaned_data

Generated Files:
data_quality_report.csv
olist_category_translation_clean.csv
olist_customers_clean.csv
olist_geolocation_clean.csv
olist_order_items_clean.csv
olist_order_payments_clean.csv
olist_order_reviews_clean.csv
olist_orders_clean.csv
olist_products_clean.csv
olist_sellers_clean.csv
